In [ ]:
import pandas as pd
import requests
import torch
import pickle

from torch.utils.data import TensorDataset, DataLoader


# 1. Koordinaten einlesen

stations = pd.read_csv(
    "/Users/dewitt/Desktop/Simulationen/"
    "git/ppython/ppython/weather.csv",
    encoding="utf-8-sig"
)


# 2. API-Request

daily_variables = [
    "temperature_2m_max",
    "relative_humidity_2m_mean",
    "wind_speed_10m_mean"
]

params = {
    "latitude": ",".join(
        stations["Lat"].astype(str)
    ),
    "longitude": ",".join(
        stations["Lon"].astype(str)
    ),
    "start_date": "2011-01-01",
    "end_date": "2025-12-31",
    "daily": ",".join(daily_variables),
    "timezone": "Europe/Berlin"
}

response = requests.get(
    "https://archive-api.open-meteo.com/"
    "v1/archive",
    params=params,
    timeout=60
)

response.raise_for_status()
weather_data = response.json()


# 3. Hilfsfunktion

def create_tensor(variable_name):

    tensor = torch.tensor(
        [
            location["daily"][variable_name]
            for location in weather_data
        ],
        dtype=torch.float32
    )

    # [Stationen, Tage] → [Tage, Stationen]
    return tensor.T


# 4. Einzelne Wettervariablen

temperatures = create_tensor(
    "temperature_2m_max"
)

humidity = create_tensor(
    "relative_humidity_2m_mean"
)

wind_speed = create_tensor(
    "wind_speed_10m_mean"
)


# 5. Features kombinieren

# Station 1 bleibt unsere Target-Station.
# Als Input verwenden wir Stationen 2–21.

# Form:
# [Tage, 20 Stationen, 3 Messwerte]

features = torch.stack(
    [
        temperatures[:, 1:],
        humidity[:, 1:],
        wind_speed[:, 1:]
    ],
    dim=2
)

history_length = 5

X_values = []
y_values = []

for target_day in range(
    history_length,
    len(features)
):
    # Die fünf Tage vor dem Zieltag
    feature_window = features[
        target_day - history_length:
        target_day
    ]

    # [5, 20, 3] → [300]
    feature_window = (
        feature_window.flatten()
    )

    X_values.append(feature_window)

    # Temperatur der Target-Station
    # am vorherzusagenden Tag
    target_temperature = temperatures[
        target_day,
        0
    ]

    y_values.append(target_temperature)


X = torch.stack(X_values)

y = (
    torch.stack(y_values)
    .unsqueeze(1)
)

print("Features vor Flatten:", features.shape)
print("X:", X.shape)
print("y:", y.shape)

with open("weather_loader.pkl", "wb") as file:
    pickle.dump(data_loader, file)



In [ ]:
# 6. Test- und Training-Data trennen

split = int(len(X) * 0.8)

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)


In [ ]:
# 7. NN Definition

class WeatherModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(300, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )


    def forward(self, x):
        return self.network(x)


In [ ]:
# 8. Loss-Funktion, Optimizer und NN initialisieren

model = WeatherModel()

loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

In [ ]:
# 9. Train-Loop

for epoch in range(5000):
    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:
        prediction = model(X_batch)
        loss = loss_function(prediction, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch}: "
            f"Loss = {total_loss / len(train_loader):.2f}"
        )